In [1]:
# 0) Install dependencies
!pip -q install evaluate sacrebleu wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00


In [2]:
# 1) Imports
import os, re, json, random, collections, math
import wandb
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
import evaluate
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [3]:
print(torch.__version__)
print(torch.version.cuda)
!nvidia-smi

2.10.0+cu128
12.8
Wed Jun 10 07:13:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------

In [4]:
# 2) Reproducibility
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [5]:
# 3) Configuration
@dataclass
class CFG:
    # ── Model ──────────────────────────────────────────────────────
    model_name: str = "t5-base"
    device: torch.device = field(
        default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )

    # ── Data ───────────────────────────────────────────────────────
    # data_dir: str = "/kaggle/input/datasets/solanadh/aogec-dataset-with-error-type-v02"
    data_dir: str = "/kaggle/input/datasets/solanadh/aogec-dataset-with-error-type-v03"

    # ── Sequence lengths ───────────────────────────────────────────
    max_input_len:  int = 128
    max_target_len: int = 160

    # ── Optimisation ──────────────────────────────────────────────
    batch_size:       int   = 8
    grad_accum_steps: int   = 4    
    
    lr_encoder:       float = 2e-5
    lr_decoder:       float = 3e-5
    
    weight_decay:     float = 0.01
    warmup_ratio:     float = 0.06
    grad_clip:        float = 1.0
    num_workers:      int   = 2
    label_smoothing:  float = 0.05       

    # ── Mixed precision ────────────────────────────────────────────
    amp_dtype: str = "bf16"

    # ── Gradient checkpointing ────────────────────────────────────
    grad_ckpt: bool = True

    # ── Training (single stage) ───────────────────────────────────
    num_epochs: int = 1         

    # ── Early stopping ─────────────────────────────────────────────
    patience: int = 5

    # ── Generation ────────────────────────────────────────────────
    num_beams_train:    int   = 1      
    num_beams_test:     int   = 4      
    min_length:         int   = 5
    repetition_penalty: float = 1.2
    no_repeat_ngram:    int   = 3
    length_penalty:     float = 0.8

    # ── GLEU ──────────────────────────────────────────────────────
    gleu_n: int = 4

    # ── Output ────────────────────────────────────────────────────
    output_dir: str = "/kaggle/working/aogec_gec_baseline_output"
    best_ckpt:  str = ""

    def __post_init__(self):
        self.train_path = f"{self.data_dir}/AOGEC_Dataset_Labeled_train.csv"
        self.val_path   = f"{self.data_dir}/AOGEC_Dataset_Labeled_eval.csv"
        self.test_path  = f"{self.data_dir}/AOGEC_Dataset_Labeled_test.csv"
        self.best_ckpt  = os.path.join(self.output_dir, "best_model.pt")
        os.makedirs(self.output_dir, exist_ok=True)
        print(f"Device : {self.device}")
        if self.device.type == "cuda":
            print(f"GPU    : {torch.cuda.get_device_name(0)}")
            if not torch.cuda.is_bf16_supported():
                self.amp_dtype = "fp16"
                print("GPU does not support bf16 → switched to fp16")
        else:
            self.amp_dtype = "none"
            self.grad_ckpt = False
            print("CPU mode — AMP and grad-ckpt disabled")
        print(f"AMP    : {self.amp_dtype}")

cfg = CFG()

_AMP_ENABLED = cfg.amp_dtype != "none"
_AMP_DTYPE   = torch.bfloat16 if cfg.amp_dtype == "bf16" else torch.float16
_autocast    = lambda: torch.amp.autocast(device_type="cuda", dtype=_AMP_DTYPE, enabled=_AMP_ENABLED)
scaler       = GradScaler(enabled=(_AMP_ENABLED and cfg.amp_dtype == "fp16"))

Device : cuda
GPU    : Tesla T4
AMP    : bf16


/tmp/ipykernel_58/3211977147.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler       = GradScaler(enabled=(_AMP_ENABLED and cfg.amp_dtype == "fp16"))


In [6]:
# 4) GLEU Implementation
class GLEUScorer:
    """
    Corpus-level GLEU following Napoles et al. (2015).
    """

    def __init__(self, n: int = 4):
        self.n = n

    @staticmethod
    def _tok(text: str) -> List[str]:
        return text.strip().split()

    @staticmethod
    def _ngrams(tokens: List[str], n: int) -> collections.Counter:
        return collections.Counter(
            tuple(tokens[i: i + n])
            for i in range(max(0, len(tokens) - n + 1))
        )

    def _sent_gleu(self, src: List[str], hyp: List[str], refs: List[List[str]]) -> float:
        if not hyp:
            return 0.0
        best = 0.0
        for ref in refs:
            total_match = total_possible = 0
            for k in range(1, self.n + 1):
                hyp_ng = self._ngrams(hyp, k)
                ref_ng = self._ngrams(ref, k)
                total_match    += sum(min(c, ref_ng[g]) for g, c in hyp_ng.items())
                total_possible += max(len(hyp) - k + 1, len(ref) - k + 1, 1)
            score = (total_match / total_possible) if total_possible else 0.0
            if hyp == src and hyp != ref:
                score = 0.0
            best = max(best, score)
        return best

    def corpus_gleu(self, sources: List[str], hypotheses: List[str],
                    references: List[List[str]]) -> float:
        assert len(sources) == len(hypotheses) == len(references)
        if not hypotheses:
            return 0.0
        total = sum(
            self._sent_gleu(self._tok(s), self._tok(h), [self._tok(r) for r in rs])
            for s, h, rs in zip(sources, hypotheses, references)
        )
        return round(total / len(hypotheses) * 100.0, 4)

    def sentence_gleu(self, source: str, hypothesis: str, references: List[str]) -> float:
        return self._sent_gleu(
            self._tok(source), self._tok(hypothesis),
            [self._tok(r) for r in references],
        ) * 100.0


gleu_scorer = GLEUScorer(n=cfg.gleu_n)
chrf_metric = evaluate.load("chrf")
print("✅ GLEU scorer and chrF metric ready.")

✅ GLEU scorer and chrF metric ready.


In [7]:
# 5) Data Preprocessing

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s([?.!,;:])", r"\1", text)
    return text.strip()

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    required = ["input", "target", "error_type"]
    
    # REMOVE gender-agreement-error
    df = df[df["error_type"] != "gender-agreement-error"]
    
    df = df[required].copy()
    df.dropna(subset=required, inplace=True)
    for col in required:
        df[col] = df[col].astype(str).apply(normalize_text)
    
    df.drop_duplicates(subset=required, inplace=True)
    return df.reset_index(drop=True)

train_df = clean_df(pd.read_csv(cfg.train_path, encoding="utf-8", encoding_errors="ignore"))
val_df   = clean_df(pd.read_csv(cfg.val_path,   encoding="utf-8", encoding_errors="ignore"))
test_df  = clean_df(pd.read_csv(cfg.test_path,  encoding="utf-8", encoding_errors="ignore"))

print(f"✅ Data | Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")

✅ Data | Train=12673  Val=1598  Test=1588


In [8]:
# 6) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
_PAD_ID   = tokenizer.pad_token_id

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# 7) Prompt helper.

def build_prompt(text: str) -> str:
    """
    Single fixed prompt for the GEC-only model.
    No error-type label is injected — the decoder must correct
    the sentence from linguistic context alone.
    """
    return f"correct afan oromo grammar: {text}"

def post_process(text: str) -> str:
    text = re.sub(r'\s([?.!,;:](?:\s|$))', r'\1', text)
    return text.strip()
    
def normalize_for_eval(text: str) -> str:
    """
    Applied to BOTH hypothesis and reference just before GLEU/chrF scoring.
    Guarantees both sides go through identical normalisation regardless of
    whether text came from the CSV or model output.
    """
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([?.!,;:])', r'\1', text)
    return text.strip()

In [10]:
# 8) Dataset

class GECDataset(Dataset):
    """
    Pre-tokenises the correction prompt and decoder target for each sample.
    Returns input_ids, attention_mask, and decoder labels.
    """

    def __init__(self, df: pd.DataFrame, tokenizer, max_in: int, max_out: int):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_in    = max_in
        self.max_out   = max_out
        self._build()

    def _encode(self, text: str, max_len: int) -> Dict[str, torch.Tensor]:
        enc = self.tokenizer(
            text, max_length=max_len, truncation=True,
            padding="max_length", return_tensors="pt",
        )
        return {k: v.squeeze(0) for k, v in enc.items()}

    def _build(self):
        self.input_ids       = []
        self.attention_masks = []
        self.decoder_labels  = []

        for i in tqdm(range(len(self.df)), desc="Tokenising", leave=False):
            row    = self.df.iloc[i]
            p_enc  = self._encode(build_prompt(str(row["input"])), self.max_in)
            t_enc  = self._encode(str(row["target"]),              self.max_out)

            lbl = t_enc["input_ids"].clone()
            lbl[lbl == _PAD_ID] = -100

            self.input_ids.append(p_enc["input_ids"])
            self.attention_masks.append(p_enc["attention_mask"])
            self.decoder_labels.append(lbl)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "labels":         self.decoder_labels[idx],
            "raw_input":      row["input"],
            "raw_target":     row["target"],
        }


train_ds = GECDataset(train_df, tokenizer, cfg.max_input_len, cfg.max_target_len)
val_ds   = GECDataset(val_df,   tokenizer, cfg.max_input_len, cfg.max_target_len)
test_ds  = GECDataset(test_df,  tokenizer, cfg.max_input_len, cfg.max_target_len)
print("✅ Datasets ready.")

Tokenising:   0%|          | 0/12673 [00:00<?, ?it/s]

Tokenising:   0%|          | 0/1598 [00:00<?, ?it/s]

Tokenising:   0%|          | 0/1588 [00:00<?, ?it/s]

✅ Datasets ready.


In [11]:
# 9) DataLoaders
_loader_kwargs = dict(
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.device.type == "cuda"),
    persistent_workers=(cfg.num_workers > 0),
    prefetch_factor=(2 if cfg.num_workers > 0 else None),
)
train_loader = DataLoader(train_ds, shuffle=False,  drop_last=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)
print(f"DataLoaders ready | {len(train_loader)} train batches")

DataLoaders ready | 1584 train batches


In [12]:
# 10) Model

class T5GEC(nn.Module):
    """
    Seq2seq T5 model for Afan Oromo grammatical error correction.

    Architecture
    ─────────────
    correction_prompt → T5 Encoder → T5 Decoder (teacher-forced) → gen_loss
    correction_prompt → T5 Encoder → T5 Decoder (beam/greedy)    → corrected text

    """

    def __init__(self, model_name: str, grad_ckpt: bool = False):
        super().__init__()
        self.seq2seq = T5ForConditionalGeneration.from_pretrained(model_name)

        if grad_ckpt:
            self.seq2seq.encoder.gradient_checkpointing_enable()
            print("✅ Gradient checkpointing enabled on encoder.")

    def forward(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        labels:         torch.Tensor,
    ) -> torch.Tensor:
        """
        Single-pass forward. Returns gen_loss (scalar).
        """
        out = self.seq2seq(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True,
        )
        return out.loss

    @torch.no_grad()
    def generate(
        self,
        input_ids:      torch.Tensor,
        attention_mask: torch.Tensor,
        cfg,
        num_beams:      int = 1,
    ) -> List[str]:
        """
        Batched generation. Returns decoded strings.
        """
        out_ids = self.seq2seq.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=cfg.max_target_len,
            min_length=cfg.min_length,
            num_beams=num_beams,
            repetition_penalty=cfg.repetition_penalty,
            no_repeat_ngram_size=cfg.no_repeat_ngram,
            length_penalty=cfg.length_penalty,
            early_stopping=(num_beams > 1),
        )
        return [
            post_process(tokenizer.decode(ids, skip_special_tokens=True))
            for ids in out_ids
        ]

In [13]:
# 11) Model initialisation

model = T5GEC(
    cfg.model_name,
    grad_ckpt=cfg.grad_ckpt,
).to(cfg.device)

print(f"✅ Model ready | params={sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Gradient checkpointing enabled on encoder.
✅ Model ready | params=222,903,552


In [14]:
# 12) Optimizer + Scheduler

def build_optimizer_and_scheduler(model, n_batches: int, n_epochs: int):
    enc_params = []
    dec_params = []
    seen = set()

    def _add(p, bucket):
        if id(p) not in seen and p.requires_grad:
            seen.add(id(p)); bucket.append(p)

    for name, param in model.named_parameters():
        if "decoder" in name or "lm_head" in name:
            _add(param, dec_params)
        else:
            _add(param, enc_params)

    groups = [
        {"params": enc_params, "lr": cfg.lr_encoder},
        {"params": dec_params, "lr": cfg.lr_decoder},
    ]
    groups = [g for g in groups if g["params"]]

    optimizer = AdamW(groups, weight_decay=cfg.weight_decay)

    effective_steps = math.ceil(n_batches / cfg.grad_accum_steps)
    total_steps     = effective_steps * n_epochs
    warmup_steps    = max(1, int(cfg.warmup_ratio * total_steps))

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    print(f"  Optimizer | total_steps={total_steps} warmup={warmup_steps} "
          f"| LRs={[g['lr'] for g in groups]}")
    return optimizer, scheduler

optimizer, scheduler = build_optimizer_and_scheduler(
    model, len(train_loader), cfg.num_epochs
)

  Optimizer | total_steps=396 warmup=23 | LRs=[0.0, 0.0]


In [15]:
# 13) Training loop

def train_one_epoch(
    model, loader, optimizer, scheduler, scaler
) -> float:
    """Returns mean generation loss for the epoch."""
    model.train()
    run_gen = 0.0
    optimizer.zero_grad()
    n_steps = len(loader)

    for step, batch in enumerate(tqdm(loader, desc="  Train", leave=False)):
        ids  = batch["input_ids"].to(cfg.device, non_blocking=True)
        mask = batch["attention_mask"].to(cfg.device, non_blocking=True)
        lbl  = batch["labels"].to(cfg.device, non_blocking=True)

        with _autocast():
            gen_loss = model(ids, mask, lbl)       # single forward pass

        scaled = scaler.scale(gen_loss / cfg.grad_accum_steps)
        scaled.backward()
        run_gen += gen_loss.item()

        # Log batch loss every X steps (e.g., every 10 steps)
        if step % 10 == 0:
            wandb.log({"train/batch_loss": gen_loss.item()})

        is_last = (step == n_steps - 1)
        if (step + 1) % cfg.grad_accum_steps == 0 or is_last:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

    return run_gen / n_steps

In [16]:
# 14) Validation loop

@torch.no_grad()
def validate(
    model, loader, num_beams: int = 1
) -> Dict[str, Any]:
    model.eval()
    val_gen = 0.0
    all_src, all_hyp, all_ref = [], [], []

    for batch in tqdm(loader, desc="  Val  ", leave=False):
        ids      = batch["input_ids"].to(cfg.device, non_blocking=True)
        mask     = batch["attention_mask"].to(cfg.device, non_blocking=True)
        lbl      = batch["labels"].to(cfg.device, non_blocking=True)
        raw_ins  = list(batch["raw_input"])
        raw_tgts = list(batch["raw_target"])

        with _autocast():
            gen_loss  = model(ids, mask, lbl)
            gen_texts = model.generate(ids, mask, cfg, num_beams=num_beams)

        val_gen += gen_loss.item()
        all_src.extend(raw_ins)
        all_hyp.extend(gen_texts)
        all_ref.extend(raw_tgts)

    norm_hyp = [normalize_for_eval(h) for h in all_hyp]
    norm_ref = [normalize_for_eval(r) for r in all_ref]
    norm_src = [normalize_for_eval(s) for s in all_src]
    
    gleu = gleu_scorer.corpus_gleu(norm_src, norm_hyp, [[r] for r in norm_ref])
    chrf = chrf_metric.compute(predictions=norm_hyp, references=norm_ref)["score"]

    return {
        "gen_loss": val_gen / len(loader),
        "gleu":     gleu,
        "chrf":     chrf,
    }

In [17]:
# 15) Checkpoint helpers

def save_ckpt(model, epoch: int, gleu: float):
    torch.save({
        "model_state_dict": model.state_dict(),
        "best_gleu":        gleu,
        "epoch":            epoch,
    }, cfg.best_ckpt)

In [18]:
# 16) Training history + plots

history: List[Dict] = []

def save_plots():
    if not history:
        return
    df_h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("AOGEC GEC Baseline — Training History",
                 fontsize=13, fontweight="bold")

    axes[0].plot(df_h["epoch"], df_h["train_gen"], marker="o", label="Train")
    axes[0].plot(df_h["epoch"], df_h["val_gen_loss"], marker="s", label="Val")
    axes[0].set_title("Generation Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(df_h["epoch"], df_h["val_gleu"], marker="o", color="green")
    axes[1].set_title("Validation GLEU"); axes[1].set_xlabel("Epoch")

    axes[2].plot(df_h["epoch"], df_h["val_chrf"], marker="o", color="orange")
    axes[2].set_title("Validation chrF"); axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    path = os.path.join(cfg.output_dir, "training_history.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  📊 Training plot saved → {path}")

In [19]:
# 17) Single-stage training

best_gleu        = -1.0
patience_counter =  0

# --- Initialize W&B Run ---
wandb.init(
    project="AOGEC-GEC-Baseline",
    name=f"t5-base-run-{cfg.amp_dtype}",
    config=cfg.__dict__
)

print(f"\n{'='*60}")
print(f"Training GEC baseline  ({cfg.num_epochs} epochs max)")
print(f"{'='*60}")

for epoch in range(1, cfg.num_epochs + 1):
    train_gen = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
    val       = validate(model, val_loader, num_beams=cfg.num_beams_train)

    history.append({
        "epoch":        epoch,
        "train_gen":    train_gen,
        "val_gen_loss": val["gen_loss"],
        "val_gleu":     val["gleu"],
        "val_chrf":     val["chrf"],
    })

    print(f"[E{epoch:02d}/{cfg.num_epochs}] "
          f"train_gen={train_gen:.4f}  "
          f"val_gen={val['gen_loss']:.4f}  "
          f"GLEU={val['gleu']:.2f}  "
          f"chrF={val['chrf']:.2f}")

    # ── Log to W&B ───────────────────────────────────────────
    wandb.log({
        "epoch": epoch,
        "train/generation_loss": train_gen,
        "val/generation_loss": val["gen_loss"],
        "val/GLEU": val["gleu"],
        "val/chrF": val["chrf"]
    })
    # ─────────────────────────────────────────────────────────

    if val["gleu"] > best_gleu:
        best_gleu = val["gleu"]
        patience_counter = 0
        save_ckpt(model, epoch, best_gleu)
        print(f"  → Checkpoint saved  (GLEU={best_gleu:.2f})")
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{cfg.patience})")
        if patience_counter >= cfg.patience:
            print("  → Early stopping."); break

# Save history JSON
history_path = os.path.join(cfg.output_dir, "training_history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

save_plots()
print(f"\nTraining complete. History → {history_path}")

# ── Finish W&B Run ────────────────────────────────────────
wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: solenabera55 (solanabera) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Training GEC baseline  (1 epochs max)


  Train:   0%|          | 0/1584 [00:00<?, ?it/s]

  Val  :   0%|          | 0/200 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[E01/1] train_gen=0.4665  val_gen=0.3288  GLEU=14.62  chrF=86.87
  → Checkpoint saved  (GLEU=14.62)
  📊 Training plot saved → /kaggle/working/aogec_gec_baseline_output/training_history.png

Training complete. History → /kaggle/working/aogec_gec_baseline_output/training_history.json


epoch,▁
train/batch_loss,▅██▇▅█▅▃▂▂▅▃▃▄▃▃▄▃▃▃▁▃▂▂▂▃▄▂▁▁▃▂▃▃▂▁▂▂▄▃
train/generation_loss,▁
val/GLEU,▁
val/chrF,▁
val/generation_loss,▁
epoch,1
train/batch_loss,0.31812
train/generation_loss,0.46653
val/GLEU,14.6243
val/chrF,86.8689


In [20]:
# 18) Final Test Evaluation

ckpt = torch.load(cfg.best_ckpt, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model = model.to(cfg.device).eval()

print(f"\nBest model: Epoch {ckpt['epoch']} | GLEU={ckpt['best_gleu']:.2f}")

all_src, all_ref, all_hyp = [], [], []
rows = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        ids      = batch["input_ids"].to(cfg.device, non_blocking=True)
        mask     = batch["attention_mask"].to(cfg.device, non_blocking=True)
        raw_ins  = list(batch["raw_input"])
        # raw_tgts = [post_process(t) for t in batch["raw_target"]]
        raw_tgts = [normalize_for_eval(t) for t in batch["raw_target"]]

        with _autocast():
            gen_texts = model.generate(
                ids, mask, cfg,
                num_beams=cfg.num_beams_test,    # beam search at test time
            )

        all_src.extend(raw_ins)
        all_ref.extend(raw_tgts)
        all_hyp.extend(gen_texts)
        for inp, tgt, hyp in zip(raw_ins, raw_tgts, gen_texts):
            rows.append([inp, tgt, hyp])

# ── Metrics ───────────────────────────────────────────────────────────
test_gleu = gleu_scorer.corpus_gleu(all_src, all_hyp, [[r] for r in all_ref])
test_chrf = chrf_metric.compute(predictions=all_hyp, references=all_ref)["score"]

print("\n" + "─" * 50)
print("GENERATION PERFORMANCE (test set)")
print(f"  GLEU  : {test_gleu:.2f}")
print(f"  chrF  : {test_chrf:.2f}")

# ── Save predictions CSV ──────────────────────────────────────────────
rows_df  = pd.DataFrame(rows, columns=["input", "target", "prediction"])
out_path = os.path.join(cfg.output_dir, "test_predictions.csv")
rows_df.to_csv(out_path, index=False)
print(f"\nPredictions CSV → {out_path}")


Best model: Epoch 1 | GLEU=14.62


Testing:   0%|          | 0/199 [00:00<?, ?it/s]


──────────────────────────────────────────────────
GENERATION PERFORMANCE (test set)
  GLEU  : 13.47
  chrF  : 86.03

Predictions CSV → /kaggle/working/aogec_gec_baseline_output/test_predictions.csv


In [21]:
# 19) Single-sentence inference

@torch.no_grad()
def predict_single(text: str, num_beams: int = 4) -> str:
    """Generate a corrected sentence for a single Afan Oromo input."""
    model.eval()
    text = normalize_for_eval(text)
    enc = tokenizer(
        build_prompt(text),
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_input_len,
        padding="max_length",
    ).to(cfg.device)

    with _autocast():
        corrections = model.generate(
            enc["input_ids"], enc["attention_mask"],
            cfg, num_beams=num_beams,
        )
    return corrections[0]


# ── Demo ──────────────────────────────────────────────────────────────
demo_samples = [
    "Abubakir fi Galgaloon bara kana barataa cimoo jedhamuu badhaafame ture .",
    "Inistiitiyuutiin Siinqee dubartootaa guyyaa idil addunyaa kabaje .",
    "Waan hundumaa darbuuf barbaachisa obsa .",
    "Ijaarani jiru isaan mana haaraa.",
    "Isheen kolfitee ture.",
]

print("\n" + "─" * 65)
print("DEMO — Afan Oromo GEC Baseline")
print("─" * 65)
for text in demo_samples:
    correction = predict_single(text)
    print(f"Input      : {text}")
    print(f"Correction : {correction}")
    print("─" * 65)


─────────────────────────────────────────────────────────────────
DEMO — Afan Oromo GEC Baseline
─────────────────────────────────────────────────────────────────
Input      : Abubakir fi Galgaloon bara kana barataa cimoo jedhamuu badhaafame ture .
Correction : abubakir fi galgaloon bara kana barataa cimoo jedhamuu badhaafame ture.
─────────────────────────────────────────────────────────────────
Input      : Inistiitiyuutiin Siinqee dubartootaa guyyaa idil addunyaa kabaje .
Correction : inistiitiyuutiin siinqee dubartootaa guyyaa idil addunyaan kabaje.
─────────────────────────────────────────────────────────────────
Input      : Waan hundumaa darbuuf barbaachisa obsa .
Correction : waan hundumaa darbuuf barbaachisa obsa.
─────────────────────────────────────────────────────────────────
Input      : Ijaarani jiru isaan mana haaraa.
Correction : ijaarani jiru isaan mana haaraa.
─────────────────────────────────────────────────────────────────
Input      : Isheen kolfitee ture.
Correct